In [6]:
import duckdb
import os

# -----------------------------
# CONNECT TO DATABASE
# -----------------------------
conn = duckdb.connect(
    "../database/airport_ops.duckdb"
)

# -----------------------------
# CREATE GOLD TABLE
# -----------------------------
conn.execute("""

CREATE OR REPLACE TABLE
gold_airport_dashboard AS

SELECT

    a.iata_code,
    a.airport_name,
    a.municipality,
    a.iso_country,

    a.nearby_flights,

    a.temperature_c,
    a.wind_kph,
    a.visibility_km,
    a.weather_condition,

    a.weather_risk,
    a.airport_stress_score,
    a.latitude_deg,
    a.longitude_deg,

    CASE

        WHEN a.airport_stress_score >= 100
        THEN 'Critical'

        WHEN a.airport_stress_score >= 50
        THEN 'High'

        WHEN a.airport_stress_score >= 20
        THEN 'Medium'

        ELSE 'Low'

    END
    AS stress_level,

    -- Delay prediction fields
    d.delay_probability,
    d.delay_risk_level,
    d.likely_delay_reason
             
FROM airport_metrics a

LEFT JOIN delay_prediction d
ON a.airport_name =
d.airport_name

""")

print("Gold table created!")

# -----------------------------
# PREVIEW DATA
# -----------------------------
preview = conn.sql("""
SELECT *
FROM gold_airport_dashboard
LIMIT 20
""").df()

print(preview)

# -----------------------------
# CREATE EXPORT FOLDER
# -----------------------------
os.makedirs(
    "../exports",
    exist_ok=True
)

# -----------------------------
# EXPORT PARQUET
# -----------------------------
gold = conn.sql("""
SELECT *
FROM gold_airport_dashboard
""").df()

gold.to_parquet(
    "../exports/gold_airport_dashboard.parquet",
    engine="pyarrow",
    compression="snappy",
    index=False
)

print(
    "Parquet exported successfully!"
)

# -----------------------------
# CLOSE CONNECTION
# -----------------------------
conn.close()

print("Done!")

Gold table created!
   iata_code                                     airport_name    municipality  \
0        PRN      Priština Adem Jashari International Airport       Prishtina   
1        YAZ                      Tofino / Long Beach Airport          Tofino   
2        QBC                              Bella Coola Airport     Bella Coola   
3        YBG                      Saguenay-Bagotville Airport        Saguenay   
4        YBL                           Campbell River Airport  Campbell River   
5        YCD                                  Nanaimo Airport         Nanaimo   
6        YCE    Centralia / James T. Field Memorial Aerodrome      Huron Park   
7        YCG         Castlegar/West Kootenay Regional Airport       Castlegar   
8        YEG                   Edmonton International Airport        Edmonton   
9        YEN                                  Estevan Airport         Estevan   
10       YTM             Mont-Tremblant International Airport       La Macaza   
11      